# Baselines: Base y COT

Columnas **Base** y **COT** de la Tabla 1 de LatentR3: Qwen2.5-1.5B sin ajustar, que escribe el título del siguiente ítem con decodificación restringida al catálogo (10 beams, trie).
- **Base:** el prompt de D3/LatentR3 sin token latente, el mismo de BIGRec.
- **COT:** el prompt de Base más ratings, frases de razonamiento de Rec-SAVER y `### Reason: `. Etapa 1: razonamiento greedy; etapa 2: título condicionado al razonamiento.

No hay entrenamiento. Escribe predicciones en `checkpoints/base_cot/<categoría>/` (no versionado) y puede retomarse. Se ejecuta desde `notebooks/base_cot/`. Las decisiones, los costos y los resultados están en [`README.md`](README.md).

## Entorno y configuración

In [1]:
import os
try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False
if not IN_COLAB:
    os.environ['HF_HUB_OFFLINE'] = '1'   # Qwen2.5-1.5B ya está en la caché local: no se descarga nada
    # Con una GPU integrada, PyTorch ve dos GPUs. Se deja visible solo la dedicada (índice 0). Tiene que ir antes
    # de importar torch; no se combina con HIP_VISIBLE_DEVICES (`../../README.md`)
    os.environ.setdefault('CUDA_VISIBLE_DEVICES', '0')

import gc
import json
import shutil
import pandas as pd
import torch
import transformers

E-001h rocSHMEM Could not open libnuma. Returning	NUMAWrapper@src/gda/numa_wrapper.cpp:48


In [2]:
names = {'CDs_and_Vinyl': 'cd_data',
         'Musical_Instruments': 'music_data',
         'Toys_and_Games': 'toys_data',
         'Video_Games': 'games_data'}
categorias = {nombre: categoria for categoria, nombre in names.items()}
DATASETS_BASE = ['toys_data', 'cd_data', 'games_data', 'music_data']   # Base es barato: por defecto los 4
DATASETS_COT = ['toys_data']   # COT cuesta más: se fija con lo medido en la prueba rápida
MAX_TOKENS_RAZON = 256         # tope del razonamiento (decisión del proyecto)
SAMPLE_TEST = -1               # -1: test completo. >0: muestra fija de filas (seed 0), una desviación a documentar

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
import sys
base = 'data_amazon_2018' if IN_COLAB else os.path.join('..', '..', 'repo_latentr3', 'data')
codigo_d3 = os.path.abspath('DecodingMatters' if IN_COLAB else os.path.join('..', '..', 'DecodingMatters'))
latentr3 = os.path.abspath('repo_latentr3' if IN_COLAB else os.path.join('..', '..', 'repo_latentr3'))
SALIDA = ('/content/drive/MyDrive/IIC3633/checkpoints/base_cot' if IN_COLAB
          else os.path.abspath(os.path.join('..', '..', 'checkpoints', 'base_cot')))
SALIDA_BIGREC = ('/content/drive/MyDrive/IIC3633/checkpoints/bigrec' if IN_COLAB
                 else os.path.abspath(os.path.join('..', '..', 'checkpoints', 'bigrec')))
calc_py = os.path.join(latentr3, 'src', 'utils', 'calc.py')
# Carpeta de las predicciones de cada método: el nombre lleva el tope y la muestra, así que cambiar la
# configuración no mezcla partes de corridas distintas
sufijo = f'_n{SAMPLE_TEST}' if SAMPLE_TEST > 0 else ''
CARPETA = {'base': 'base' + sufijo, 'cot': f'cot_{MAX_TOKENS_RAZON}' + sufijo}

In [3]:
for archivo in ['code/dataset.py', 'code/LogitProcesser.py']:
    if not os.path.isfile(os.path.join(codigo_d3, archivo)):
        raise FileNotFoundError(f'No se encontró {archivo} en {codigo_d3}. Descargar '
                                'https://github.com/SAI990323/DecodingMatters en esa carpeta y hacer checkout del commit 585c842.')
if not os.path.isfile(calc_py):
    raise FileNotFoundError(f'No se encontró {calc_py}. En Colab hay que subir repo_latentr3/src/.')
# base_cot_baseline.py agrega a sys.path notebooks/ (baseline_comun.py) y notebooks/bigrec/ (bigrec_baseline.py)
import base_cot_baseline as bc
bb = bc.bb
bb.cargar_d3(codigo_d3)   # verifica el sha256 antes de importar

print('torch', torch.__version__, '| transformers', transformers.__version__)
if transformers.__version__ != '4.51.3':
    print('AVISO: los autores usan transformers 4.51.3 y advierten que otra versión cambia los resultados (issue #2).')
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0), '| bf16:', torch.cuda.is_bf16_supported())
else:
    print('Sin GPU: solo se pueden correr las verificaciones sin GPU.')

torch 2.14.1+rocm7.14 | transformers 4.51.3
AMD Radeon RX 9070 XT | bf16: True


## Datos y prompts

Los mismos CSV que los demás baselines (solo `test` y el catálogo de `info/`; no hay entrenamiento). Del historial se usan los títulos (Base y COT) y los ratings (COT). El título y los datos del objetivo nunca entran al prompt.

Abajo, un prompt de Base y uno de COT para la misma fila. En COT, todo lo que no es el rating, las frases de razonamiento ni la sección `### Reason: ` es idéntico a Base, con las erratas originales de D3 ("recetenly", "palyed", "{categoría}s").

In [4]:
conteos = {}
for nombre, categoria in categorias.items():
    with open(bb.buscar_archivo(base, 'info', categoria), encoding='utf-8') as f:
        conteos[nombre] = {'test': len(pd.read_csv(bb.buscar_archivo(base, 'test', categoria))), 'ítems': sum(1 for _ in f)}
pd.DataFrame(conteos).T

,test,ítems
cd_data,6157,5841
music_data,8313,5030
toys_data,6738,6299
games_data,9397,5308


In [5]:
tokenizer = bc.preparar_tokenizer()
categoria = 'Toys_and_Games'
archivo = bb.buscar_archivo(base, 'test', categoria)
enc_base, _ = bc.construir_base(tokenizer, archivo, categoria, sample=1)
enc_cot, _ = bc.construir_cot(tokenizer, archivo, categoria, sample=1)
print('=== Base ===\n' + tokenizer.decode(enc_base[0]['input_ids']) + '<generación restringida>')
print('\n=== COT, etapa 1 ===\n' + tokenizer.decode(enc_cot[0]['input_ids']) + '<razonamiento greedy>')
print('\n=== COT, etapa 2 (con un razonamiento de ejemplo) ===')
raiz = bc.raiz_trie(tokenizer, bb.buscar_archivo(base, 'info', categoria))
print(tokenizer.decode(bc.ids_etapa2(enc_cot[0]['input_ids'], '<razonamiento>', tokenizer, raiz)['input_ids'])
      + '<generación restringida>')

100%|██████████| 1/1 [00:00<00:00, 688.83it/s]


=== Base ===
Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request. 

### Instruction:
Given a list of toys and games the user recetenly enjoy, please write a new toys and games that the user may bought
### User Input: 
The user has palyed the following toys and gamess before: "Hasbro A4754 Perfection Game Popping Shapes and Pieces Game for Kids Ages 4 and Up, Small, Blue", "Gazillion Bubbles 2 Liter Solution", "furReal StarLily, My Magical Unicorn Interactive Plush Pet Toy, Light-up Horn, Ages 4 and Up(Amazon Exclusive)", "Fisher-Price Little People A Christmas Story"

### Response: 
<generación restringida>

=== COT, etapa 1 ===
Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request. 

### Instruction:
Given a list of toys and games the user recetenly enjoy, please write a new toys 

## Verificaciones sin GPU

En los 4 datasets, solo con el tokenizer:
- **V1, trie de Base** (`bb.verificar_trie`): todos los prompts de `test` terminan en los 4 tokens de `### Response: \n`, que son la raíz del trie, y recorrer el trie da exactamente los títulos del catálogo. Así no puede haber predicciones fuera de él.
- **V2, alineación de COT con D3** (`bc.verificar_alineacion`): sin ratings ni razonamiento, el prompt de COT da los mismos tokens que `D3Dataset` en 200 filas de `train`, `valid` y `test`. Así, lo único que COT agrega a Base es lo marcado.
- **V3, etapa 2 de COT** (`bc.verificar_etapa2`): con razonamientos de distintas formas (vacío, con saltos de línea, con espacios o comillas), el prompt de la etapa 2 termina en la raíz del trie, y el trie da el catálogo.
- **V4, fuga** (`bc.contar_recompras`): el prompt solo usa columnas `history_*`. Cuenta las filas cuyo objetivo ya está en el historial (recompras, legítimas y presentes también en Base) y comprueba que en las demás el título objetivo entre comillas no aparezca en el prompt (debe ser 0).

También se miden los tokens del prompt de Base y de la etapa 1 de COT.

In [6]:
import numpy as np
verificacion = {}
for nombre, categoria in categorias.items():
    archivo, info = bb.buscar_archivo(base, 'test', categoria), bb.buscar_archivo(base, 'info', categoria)
    enc_base, _ = bc.construir_base(tokenizer, archivo, categoria)
    enc_cot, _ = bc.construir_cot(tokenizer, archivo, categoria)
    v1 = bb.verificar_trie(info, tokenizer, enc_base)
    bc.verificar_alineacion(base, categoria, tokenizer)
    bc.verificar_etapa2(base, categoria, tokenizer)
    v4 = bc.contar_recompras(base, categoria)
    assert v4['objetivo en el prompt sin recompra'] == 0
    largo_base = np.array([len(e['input_ids']) for e in enc_base])
    largo_cot = np.array([len(e['input_ids']) for e in enc_cot])
    verificacion[nombre] = {'títulos únicos': v1['títulos únicos'], 'caminos del trie': v1['caminos del trie'],
                            'tokens del título más largo': v1['tokens del título más largo'],
                            'recompras': v4['objetivo en el historial'], '% recompras': 100 * v4['porcentaje'],
                            'tokens Base (media)': largo_base.mean(), 'tokens Base (máx.)': largo_base.max(),
                            'tokens COT etapa 1 (media)': largo_cot.mean(), 'tokens COT etapa 1 (máx.)': largo_cot.max()}
print('V1–V4 correctas en los 4 datasets')
pd.DataFrame(verificacion).T.round(1)

100%|██████████| 200/200 [00:00<00:00, 3196.73it/s]


V1–V4 correctas en los 4 datasets


,títulos únicos,caminos del trie,tokens del título más largo,recompras,% recompras,tokens Base (media),tokens Base (máx.),tokens COT etapa 1 (media),tokens COT etapa 1 (máx.)
cd_data,5377.0,5377.0,95.0,190.0,3.1,125.1,261.0,231.2,390.0
music_data,5017.0,5017.0,57.0,71.0,0.9,199.9,385.0,303.2,514.0
toys_data,6271.0,6271.0,69.0,100.0,1.5,159.2,339.0,258.7,468.0
games_data,5225.0,5225.0,68.0,199.0,2.1,151.0,297.0,253.0,426.0


## Prueba rápida

100 filas de `test` de Toys (muestra fija, seed 0), en `checkpoints/base_cot/_prueba/`, con Base y con COT con tope de 256 y de 512. El modelo se carga una vez. Mide:
- el tiempo por fila de Base y de cada etapa de COT, y la memoria;
- **V5:** que las métricas coincidan con `calc.py` y que haya 0 títulos fuera del catálogo (100 filas no alcanzan para comparar métricas con el paper);
- cómo terminan los razonamientos (marca `###`, EOS o tope), cuántos se cortan con 128, 256 y 512 tokens, y cuántos quedan vacíos;
- **V6:** que con greedy y los mismos lotes el razonamiento con tope 256 sea el prefijo del de 512. Así se justifica leer en la corrida de 512 cuántos se cortarían con 256;
- 3 razonamientos de ejemplo.

Con lo medido se estiman las horas por dataset.

In [7]:
PRUEBA = os.path.join(SALIDA, '_prueba')
categoria = 'Toys_and_Games'
info = bb.buscar_archivo(base, 'info', categoria)
modelo = bb.cargar_modelo(bc.BASE_MODEL, tokenizer)
rutas_prueba = {'base': bc.evaluar(categoria, base, PRUEBA, 'base', sample=100, modelo=modelo, tokenizer=tokenizer)}
for tope in [256, 512]:
    rutas_prueba[f'cot_{tope}'] = bc.evaluar(categoria, base, PRUEBA, 'cot', sample=100, max_tokens_razon=tope,
                                             carpeta=f'cot_{tope}', modelo=modelo, tokenizer=tokenizer)
prueba = {}
for clave, ruta in rutas_prueba.items():
    bb.verificar_con_calc(ruta, info, calc_py)   # V5: falla si las métricas no coinciden con calc.py
    r = bc.leer_registro(ruta)
    prueba[clave] = {'s por fila': r['segundos'] / r['filas'], 's por fila (razonamiento)': r['segundos_razon'] / r['filas'],
                     's por fila (título)': r['segundos_titulo'] / r['filas'], 'memoria máx. GB': r['memoria_max_gb'],
                     'lotes reducidos': len(r.get('lotes_reducidos', [])), **bb.metricas(ruta, info)}
assert all(p['fuera del catálogo'] == 0 for p in prueba.values())
pd.DataFrame(prueba).T

Sliding Window Attention is enabled but not implemented for `sdpa`; unexpected results may be encountered.
100%|██████████| 100/100 [00:00<00:00, 3176.23it/s]
[W1106 11:39:05.777050214 HIPCachingAllocator.cpp:3935] memory allocation failed with OOM on device 0 while trying to allocate 1140850688 bytes (free: 1052770304, total: 17095983104).


Toys_and_Games (base): parte 1/1, 0.31 s por fila


[W1106 11:39:41.477254118 HIPCachingAllocator.cpp:3935] memory allocation failed with OOM on device 0 while trying to allocate 1363148800 bytes (free: 1138753536, total: 17095983104).
[W1106 11:40:05.135892026 HIPCachingAllocator.cpp:3935] memory allocation failed with OOM on device 0 while trying to allocate 1398800384 bytes (free: 1285554176, total: 17095983104).
[W1106 11:40:12.133302585 HIPCachingAllocator.cpp:3935] memory allocation failed with OOM on device 0 while trying to allocate 1409286144 bytes (free: 1115684864, total: 17095983104).


Toys_and_Games (cot_256): parte 1/1, 0.75 s por fila


[W1106 11:41:20.831799898 HIPCachingAllocator.cpp:3935] memory allocation failed with OOM on device 0 while trying to allocate 1912602624 bytes (free: 1969225728, total: 17095983104).
[W1106 11:41:54.577891301 HIPCachingAllocator.cpp:3935] memory allocation failed with OOM on device 0 while trying to allocate 1950351360 bytes (free: 251658240, total: 17095983104).
[W1106 11:42:45.745504523 HIPCachingAllocator.cpp:3935] memory allocation failed with OOM on device 0 while trying to allocate 1960837120 bytes (free: 130023424, total: 17095983104).


Toys_and_Games (cot_512): parte 1/1, 1.62 s por fila


,s por fila,s por fila (razonamiento),s por fila (título),memoria máx. GB,lotes reducidos,HR@1,HR@3,HR@5,HR@10,HR@20,NDCG@1,NDCG@3,NDCG@5,NDCG@10,NDCG@20,fuera del catálogo,repetidos,filas
base,0.309126,0.000000,0.309126,8.626285,0.0,0.01,0.01,0.02,0.05,0.05,0.01,0.01,0.013869,0.023774,0.023774,0.0,0.0,100.0
cot_256,0.749861,0.204540,0.545321,10.024399,0.0,0.01,0.01,0.02,0.05,0.05,0.01,0.01,0.013869,0.023774,0.023774,0.0,0.0,100.0
cot_512,1.617251,0.440491,1.176760,12.837152,0.0,0.01,0.01,0.02,0.05,0.05,0.01,0.01,0.013869,0.023774,0.023774,0.0,0.0,100.0


In [8]:
# Cómo terminan los razonamientos y V6
display(pd.DataFrame({clave: bc.estadisticas_razon(ruta) for clave, ruta in rutas_prueba.items() if clave != 'base'}).T.round(3))
with open(rutas_prueba['cot_256']) as f:
    corto = json.load(f)
with open(rutas_prueba['cot_512']) as f:
    largo = json.load(f)
iguales = prefijos = 0
for a, b in zip(corto, largo):
    if b['tokens_generados'] <= 256:   # no se cortó en 256: el razonamiento debe ser idéntico
        # Si terminó con EOS justo en el token 256, con tope 256 el texto es el mismo pero el fin es 'tope'
        assert a['reasoning'] == b['reasoning'], (a['reasoning'], b['reasoning'])
        assert a['fin'] == b['fin'] or b['tokens_generados'] == 256, (a['fin'], b['fin'])
        iguales += 1
    else:                              # se cortó en 256: debe ser el comienzo del de 512
        assert a['fin'] == 'tope' and b['reasoning'].startswith(a['reasoning']), (a['reasoning'], b['reasoning'])
        prefijos += 1
print(f'V6: {iguales} razonamientos idénticos y {prefijos} cortados en 256 que son prefijo del de 512')

,fin: marca,fin: eos,fin: tope,vacíos,tokens generados (media),tokens generados (mediana),tokens generados (p90),tokens etapa 2 (media),cortados con tope 128,cortados con tope 256,cortados con tope 512
cot_256,0.67,0.10,0.23,0.0,150.19,128.5,256.0,410.58,0.5,0.23,NaN
cot_512,0.70,0.12,0.18,0.0,203.46,128.5,512.0,463.77,0.5,0.23,0.18


V6: 77 razonamientos idénticos y 23 cortados en 256 que son prefijo del de 512


In [9]:
print(bc.ejemplos(rutas_prueba[f'cot_{MAX_TOKENS_RAZON}'], n=3))

--- Fila 63 (marca, 82 tokens)
Historial: The user has palyed the following toys and gamess before: "Fisher-Price DoodlePro, Slim (Blue)" (rating: 5), "Little Tikes EasyScore Basketball Set" (rating: 5), "Melissa & Doug Classic Wooden Peg Puzzles, See-Inside Alphabet & Numbers, and Fish Mix & Match Colors, 3-Pack" (rating: 5), "Little Tikes Cook 'n Play Outdoor BBQ" (rating: 5), "Sesame Street Classic Plush - 3 Pcs Set - Includes Elmo, Big Bird, and Cookie Monster" (rating: 3), "Melissa & Doug Jumbo Coloring Pad - Animals" (rating: 5), "Learning Resources Pretend & Play Doctor Kit for Kids, 19 Piece set, Blue" (rating: 5), "DC Super Hero Girls Wonder Woman Shield" (rating: 4), "Jungle Gym Kingdom Pirate Ships Wheel - Blue" (rating: 4)
Razonamiento:
Based on the user's history of playing with toys and games that have high ratings, it can be inferred that the user enjoys educational and imaginative play. The user has also shown a preference for toys and games that are durable and can wit

In [10]:
# Horas por dataset con lo medido (100 filas de Toys; el largo del prompt cambia poco entre datasets)
estimacion = {'Base': bc.estimar_horas(prueba['base']['s por fila']),
              f'COT (tope {MAX_TOKENS_RAZON})': bc.estimar_horas(prueba[f'cot_{MAX_TOKENS_RAZON}']['s por fila'])}
tabla = pd.DataFrame(estimacion)
tabla.loc['Total'] = tabla.sum()
print('Referencia: BIGRec en Toys tardó 0,256 s por fila (1724 s para 6738 filas, lotes de 24).')
print(f"COT / Base en la prueba: {prueba[f'cot_{MAX_TOKENS_RAZON}']['s por fila'] / prueba['base']['s por fila']:.1f}× "
      '(el paper reporta ~25× en Toys y ~30× en Games, con lotes de 4).')
tabla.round(2)

Referencia: BIGRec en Toys tardó 0,256 s por fila (1724 s para 6738 filas, lotes de 24).
COT / Base en la prueba: 2.4× (el paper reporta ~25× en Toys y ~30× en Games, con lotes de 4).


,Base,COT (tope 256)
CDs_and_Vinyl,0.53,1.28
Musical_Instruments,0.71,1.73
Toys_and_Games,0.58,1.40
Video_Games,0.81,1.96
Total,2.63,6.37


In [11]:
del modelo
gc.collect()
torch.cuda.empty_cache()
BORRAR_PRUEBA = True
if BORRAR_PRUEBA:
    shutil.rmtree(PRUEBA, ignore_errors=True)

## Evaluación

Para cada fila de `test` se generan 10 títulos con beam search de 10 beams restringido al catálogo con el trie, ordenados por log-probabilidad dividida por el largo (`length_penalty = 1`, como BIGRec en LatentR3).
- **Base:** sobre el prompt de D3, en lotes de 24.
- **COT:** primero el razonamiento greedy, en lotes fijos de 64, hasta `###`, EOS o `MAX_TOKENS_RAZON` tokens. Después el título restringido sobre prompt + razonamiento + `### Response: \n`, en lotes de 12 (con 24 no cupo en la prueba rápida). Si un lote no cabe en la GPU se divide a la mitad y queda anotado en `registro.json`.

Se guarda por partes de 256 filas en `checkpoints/base_cot/<categoría>/predicciones/<carpeta>/`, así que se puede retomar. La carpeta lleva el tope y la muestra (`base`, `cot_256`, …). Si una carpeta ya tiene partes de otra configuración, la evaluación falla en vez de mezclarlas. Al final se escribe `final_result.json` con el formato que lee `calc.py`. Cada fila de COT guarda además el razonamiento, los tokens generados y cómo terminó.

In [12]:
pendientes = [(m, n) for m, lista in [('base', DATASETS_BASE), ('cot', DATASETS_COT)] for n in lista
              if not os.path.isfile(os.path.join(SALIDA, categorias[n], 'predicciones', CARPETA[m], 'final_result.json'))]
print('Pendientes:', pendientes or 'ninguno')
modelo = bb.cargar_modelo(bc.BASE_MODEL, tokenizer) if pendientes else None
for metodo, nombre in pendientes:
    categoria = categorias[nombre]
    bc.evaluar(categoria, base, os.path.join(SALIDA, categoria), metodo, sample=SAMPLE_TEST,
               max_tokens_razon=MAX_TOKENS_RAZON, carpeta=CARPETA[metodo], modelo=modelo, tokenizer=tokenizer)
if modelo is not None:
    del modelo
    gc.collect()
    torch.cuda.empty_cache()

Pendientes: [('base', 'toys_data'), ('base', 'cd_data'), ('base', 'games_data'), ('base', 'music_data'), ('cot', 'toys_data')]


100%|██████████| 6738/6738 [00:01<00:00, 3479.84it/s]


Toys_and_Games (base): parte 1/27, 0.26 s por fila
Toys_and_Games (base): parte 2/27, 0.26 s por fila
Toys_and_Games (base): parte 3/27, 0.24 s por fila
Toys_and_Games (base): parte 4/27, 0.25 s por fila


[W1106 11:48:10.854031149 HIPCachingAllocator.cpp:3935] memory allocation failed with OOM on device 0 while trying to allocate 1247805440 bytes (free: 486539264, total: 17095983104).


Toys_and_Games (base): parte 5/27, 0.24 s por fila
Toys_and_Games (base): parte 6/27, 0.26 s por fila
Toys_and_Games (base): parte 7/27, 0.25 s por fila
Toys_and_Games (base): parte 8/27, 0.26 s por fila
Toys_and_Games (base): parte 9/27, 0.25 s por fila
Toys_and_Games (base): parte 10/27, 0.25 s por fila


[W1106 11:54:32.201444980 HIPCachingAllocator.cpp:3935] memory allocation failed with OOM on device 0 while trying to allocate 1264582656 bytes (free: 627048448, total: 17095983104).


... (15 outputs left out)

[W1106 13:26:57.888494353 HIPCachingAllocator.cpp:3935] memory allocation failed with OOM on device 0 while trying to allocate 1644167168 bytes (free: 1587544064, total: 17095983104).


Musical_Instruments (base): parte 15/33, 0.35 s por fila
Musical_Instruments (base): parte 16/33, 0.35 s por fila
Musical_Instruments (base): parte 17/33, 0.34 s por fila
Musical_Instruments (base): parte 18/33, 0.31 s por fila
Musical_Instruments (base): parte 19/33, 0.31 s por fila
Musical_Instruments (base): parte 20/33, 0.33 s por fila


[W1106 13:35:42.004316167 HIPCachingAllocator.cpp:3935] memory allocation failed with OOM on device 0 while trying to allocate 1656750080 bytes (free: 1375731712, total: 17095983104).


Musical_Instruments (base): parte 21/33, 0.36 s por fila
Musical_Instruments (base): parte 22/33, 0.35 s por fila
Musical_Instruments (base): parte 23/33, 0.34 s por fila
Musical_Instruments (base): parte 24/33, 0.35 s por fila
Musical_Instruments (base): parte 25/33, 0.33 s por fila
Musical_Instruments (base): parte 26/33, 0.33 s por fila
Musical_Instruments (base): parte 27/33, 0.33 s por fila
Musical_Instruments (base): parte 28/33, 0.35 s por fila
Musical_Instruments (base): parte 29/33, 0.36 s por fila
Musical_Instruments (base): parte 30/33, 0.36 s por fila
Musical_Instruments (base): parte 31/33, 0.38 s por fila
Musical_Instruments (base): parte 32/33, 0.36 s por fila
Musical_Instruments (base): parte 33/33, 0.35 s por fila
Toys_and_Games (cot_256): parte 1/27, 0.70 s por fila
Toys_and_Games (cot_256): parte 2/27, 0.71 s por fila
Toys_and_Games (cot_256): parte 3/27, 0.67 s por fila
Toys_and_Games (cot_256): parte 4/27, 0.71 s por fila
Toys_and_Games (cot_256): parte 5/27, 0.67 

## Métricas

HR@K y NDCG@K con el criterio de `calc.py`: hay acierto si el título generado, normalizado, es igual al del objetivo. Para cada resultado, `bb.verificar_con_calc` corre `gao` de `calc.py` sobre el mismo `final_result.json` y falla si da algo distinto. Los títulos fuera del catálogo deben ser 0. La tabla usa todos los datasets con predicciones para la configuración actual (`CARPETA`), no solo los de las listas. Para COT se agregan las estadísticas del razonamiento.

In [13]:
resultados, razon, tiempos = {}, {}, {}
for metodo in ['base', 'cot']:
    for nombre, categoria in categorias.items():
        ruta = os.path.join(SALIDA, categoria, 'predicciones', CARPETA[metodo], 'final_result.json')
        if not os.path.isfile(ruta):
            continue
        info = bb.buscar_archivo(base, 'info', categoria)
        bb.verificar_con_calc(ruta, info, calc_py)
        resultados[(nombre, metodo)] = bb.metricas(ruta, info)
        r = bc.leer_registro(ruta)
        tiempos[(nombre, metodo)] = {'h': r['segundos'] / 3600, 's por fila': r['segundos'] / r['filas'],
                                     'memoria máx. GB': r['memoria_max_gb'], 'lotes reducidos': len(r.get('lotes_reducidos', []))}
        if metodo == 'cot':
            razon[nombre] = bc.estadisticas_razon(ruta)
print('Evaluados:', list(resultados) or 'ninguno')
assert all(r['fuera del catálogo'] == 0 for r in resultados.values())
display(pd.DataFrame(resultados).T.round(4))
display(pd.DataFrame(tiempos).T.round(3))
pd.DataFrame(razon).T.round(3)

Evaluados: [('cd_data', 'base'), ('music_data', 'base'), ('toys_data', 'base'), ('games_data', 'base'), ('toys_data', 'cot')]


,,HR@1,HR@3,HR@5,HR@10,HR@20,NDCG@1,NDCG@3,NDCG@5,NDCG@10,NDCG@20,fuera del catálogo,repetidos,filas
cd_data,base,0.0102,0.0185,0.0258,0.0372,0.0372,0.0102,0.0148,0.0178,0.0216,0.0216,0.0,0.0,6157.0
music_data,base,0.0023,0.0207,0.0396,0.0635,0.0635,0.0023,0.0124,0.0200,0.0279,0.0279,0.0,0.0,8313.0
toys_data,base,0.0080,0.0217,0.0380,0.0650,0.0650,0.0080,0.0155,0.0223,0.0310,0.0310,0.0,0.0,6738.0
games_data,base,0.0161,0.0220,0.0278,0.0399,0.0399,0.0161,0.0195,0.0218,0.0257,0.0257,0.0,0.0,9397.0
toys_data,cot,0.0085,0.0203,0.0373,0.0644,0.0644,0.0085,0.0151,0.0221,0.0308,0.0308,0.0,0.0,6738.0


,,h,s por fila,memoria máx. GB,lotes reducidos
cd_data,base,0.306,0.179,8.541,0.0
music_data,base,0.790,0.342,11.222,0.0
toys_data,base,0.486,0.260,9.467,0.0
games_data,base,0.579,0.222,9.316,0.0
toys_data,cot,1.322,0.706,10.438,0.0


,fin: marca,fin: eos,fin: tope,vacíos,tokens generados (media),tokens generados (mediana),tokens generados (p90),tokens etapa 2 (media),cortados con tope 128,cortados con tope 256
toys_data,0.673,0.085,0.242,0.0,145.986,124.0,256.0,407.877,0.475,0.242


## Comparación con el paper

- **Tabla 1, Base y COT:** una corrida determinista (Apéndice E). El paper no publica los prompts, así que la comparación mide qué tan cerca queda nuestra versión, no si es la misma.
- **Instruments:** el paper reporta NDCG > HR en Base (N@5 0.0296 > H@5 0.0154; N@10 0.0411 > H@10 0.0192) y en COT (N@5 0.0261 > H@5 0.0135; N@10 0.0452 > H@10 0.0199). Con un solo ítem relevante es imposible (NDCG@K ≤ HR@K), así que esas celdas no sirven como referencia (paper, Tabla 1).
- **CDs, Tabla 6:** repite Base y COT de la Tabla 1 y agrega COT-RL (CoT aprendido con RL).
- Como referencia, nuestro BIGRec del mismo dataset si ya está evaluado (`checkpoints/bigrec/`).

In [14]:
METRICAS = ['HR@5', 'HR@10', 'NDCG@5', 'NDCG@10']
PAPER = {('toys_data', 'base'): [0.0203, 0.0359, 0.0128, 0.0178], ('toys_data', 'cot'): [0.0261, 0.0496, 0.0153, 0.0229],
         ('cd_data', 'base'): [0.0195, 0.0252, 0.0148, 0.0167], ('cd_data', 'cot'): [0.0302, 0.0406, 0.0213, 0.0246],
         ('games_data', 'base'): [0.0236, 0.0311, 0.0190, 0.0214], ('games_data', 'cot'): [0.0120, 0.0194, 0.0082, 0.0105],
         ('music_data', 'base'): [0.0154, 0.0192, 0.0296, 0.0411], ('music_data', 'cot'): [0.0135, 0.0199, 0.0261, 0.0452]}
NOTA_PAPER = {'music_data': 'NDCG > HR: imposible, no sirve como referencia'}
TABLA6_COT_RL = [0.0297, 0.0390, 0.0202, 0.0232]
NOMBRE = {'base': 'Base', 'cot': 'COT'}
filas = []
for nombre in categorias:
    for metodo in ['base', 'cot']:
        filas.append({'dataset': nombre, 'fuente': f'Tabla 1: {NOMBRE[metodo]}', **dict(zip(METRICAS, PAPER[(nombre, metodo)])),
                      'nota': NOTA_PAPER.get(nombre, '')})
        r = resultados.get((nombre, metodo))
        filas.append({'dataset': nombre, 'fuente': f'Nuestro {NOMBRE[metodo]}',
                      **({m: r[m] for m in METRICAS} if r else {}), 'nota': '' if r else 'no evaluado'})
    if nombre == 'cd_data':
        filas.append({'dataset': nombre, 'fuente': 'Tabla 6: COT-RL', **dict(zip(METRICAS, TABLA6_COT_RL)), 'nota': ''})
    ruta_bigrec = os.path.join(SALIDA_BIGREC, categorias[nombre], 'predicciones', 'bigrec', 'final_result.json')
    if os.path.isfile(ruta_bigrec):
        r = bb.metricas(ruta_bigrec, bb.buscar_archivo(base, 'info', categorias[nombre]))
        filas.append({'dataset': nombre, 'fuente': 'Nuestro BIGRec (referencia)', **{m: r[m] for m in METRICAS}, 'nota': ''})
pd.DataFrame(filas).set_index(['dataset', 'fuente']).round(4)

HR@5   HR@10  NDCG@5  NDCG@10  \
dataset    fuente                                                         
cd_data    Tabla 1: Base                0.0195  0.0252  0.0148   0.0167   
           Nuestro Base                 0.0258  0.0372  0.0178   0.0216   
           Tabla 1: COT                 0.0302  0.0406  0.0213   0.0246   
           Nuestro COT                     NaN     NaN     NaN      NaN   
           Tabla 6: COT-RL              0.0297  0.0390  0.0202   0.0232   
music_data Tabla 1: Base                0.0154  0.0192  0.0296   0.0411   
           Nuestro Base                 0.0396  0.0635  0.0200   0.0279   
           Tabla 1: COT                 0.0135  0.0199  0.0261   0.0452   
           Nuestro COT                     NaN     NaN     NaN      NaN   
toys_data  Tabla 1: Base                0.0203  0.0359  0.0128   0.0178   
           Nuestro Base                 0.0380  0.0650  0.0223   0.0310   
           Tabla 1: COT                 0.0261  0.0496  0.0153   0.0229   
           Nuestro COT                  0.0373  0.0644  0.0221   0.0308   
           Nuestro BIGRec (referencia)  0.0770  0.1060  0.0563   0.0657   
games_data Tabla 1: Base                0.0236  0.0311  0.0190   0.0214   
           Nuestro Base                 0.0278  0.0399  0.0218   0.0257   
           Tabla 1: COT                 0.0120  0.0194  0.0082   0.0105   
           Nuestro COT                     NaN     NaN     NaN      NaN   

                                                                                  nota  
dataset    fuente                                                                       
cd_data    Tabla 1: Base                                                                
           Nuestro Base                                                                 
           Tabla 1: COT                                                                 
           Nuestro COT                                                     no evaluado  
           Tabla 6: COT-RL                                                              
music_data Tabla 1: Base                NDCG > HR: imposible, no sirve como referencia  
           Nuestro Base                                                                 
           Tabla 1: COT                 NDCG > HR: imposible, no sirve como referencia  
           Nuestro COT                                                     no evaluado  
toys_data  Tabla 1: Base                                                                
           Nuestro Base                                                                 
           Tabla 1: COT                                                                 
           Nuestro COT                                                                  
           Nuestro BIGRec (referencia)                                                  
games_data Tabla 1: Base                                                                
           Nuestro Base                                                                 
           Tabla 1: COT                                                                 
           Nuestro COT                                                     no evaluado